# Bates: stochastic volatility with jumps

Two threads from earlier weeks converge here. Heston (Week 4) gave the price a
stochastic variance, which produces a realistic term structure of volatility but
is Markovian and mean-reverting, so its smile flattens too fast at short
maturities. Merton and Kou (Week 7) added jumps, which produce steep short-dated
skew a pure diffusion cannot, but on a constant-vol diffusion they miss the term
structure and, on real SPX, could only reach the steep 29 DTE skew by degenerating
to a near one-sided fit.

Bates (1996) combines them: Heston's stochastic variance *and* Merton's jumps in
the price. The division of labour is exactly what the two earlier failures asked
for. Stochastic vol supplies the term structure; jumps supply the short-dated
tail. This notebook derives the Bates characteristic function as a composition of
the two pieces we have already built and certified, then (tomorrow) implements and
validates it, reusing the model-agnostic COS core.

## 1. The model

Under the risk-neutral measure, the price carries a stochastic-variance diffusion
plus a compound Poisson jump, and the variance follows the Heston CIR process:

$$\frac{dS_t}{S_t} = (r - q - \lambda\kappa_J)\,dt + \sqrt{v_t}\,dW_t^S + dJ_t$$

$$dv_t = \kappa_v(\theta - v_t)\,dt + \xi\sqrt{v_t}\,dW_t^v, \qquad dW_t^S\,dW_t^v = \rho\,dt$$

The jump $dJ_t$ is Merton's: arrivals at Poisson rate $\lambda$, log-jump sizes
$Y \sim \mathcal{N}(\mu_J, \delta_J^2)$.

**A notation clash to settle first.** Heston's mean-reversion speed is
conventionally $\kappa$, and the Merton jump compensator is also conventionally
$\kappa$. They are unrelated. Throughout, the Heston mean-reversion is $\kappa_v$
and the jump compensator is

$$\kappa_J = \mathbb{E}[e^Y - 1] = e^{\mu_J + \frac12\delta_J^2} - 1.$$

Confusing them in code would be a silent disaster, so the names stay distinct
($\kappa_v$, `kappa_v`; $\kappa_J$, `kappa_j`).

The full parameter set is Heston's five, $v_0, \kappa_v, \theta, \xi, \rho$, plus
Merton's three, $\lambda, \mu_J, \delta_J$. Eight parameters. That richness is the
point (it can fit both term structure and short-dated skew), but it also
foreshadows a calibration problem: jumps and diffusion both add short-dated
variance, so they are partly redundant, and the fit may not pin them separately.
That identifiability tension is a theme for the calibration day.

## 2. The characteristic function is a composition

Here the Week 7 insight pays off directly. The log-price has two *independent*
sources of randomness: the Heston diffusion-with-stochastic-vol part, and the
compound Poisson jump part. Independent random variables have characteristic
functions that multiply, because the expectation of a product of independent
things factorises:

$$\phi_{\text{Bates}}(u) = \mathbb{E}\!\left[e^{iu(X_{\text{Heston}} + X_{\text{jump}})}\right]
= \mathbb{E}\!\left[e^{iuX_{\text{Heston}}}\right]\,\mathbb{E}\!\left[e^{iuX_{\text{jump}}}\right]
= \phi_{\text{Heston}}(u)\,\phi_{\text{jump}}(u).$$

The jump factor is exactly Merton's, derived in Week 7 via conditioning on the
number of jumps (Levy-Khintchine form):

$$\phi_{\text{jump}}(u) = \exp\!\Big(\lambda\tau\big[e^{iu\mu_J - \frac12 u^2\delta_J^2} - 1\big]\Big).$$

So nothing is re-derived from scratch. Bates is Heston's characteristic function
times Merton's jump factor, and both are already coded and certified. The model
is a *composition*, not a new derivation.

## 3. The one subtlety: the compensator, and not double-counting drift

Composition is not quite "multiply the two standalone characteristic functions,"
and the reason is the drift.

Each model, standalone, carries its own drift that makes its *own* discounted
price a martingale. Our `heston_char_func` carries a drift term $r\,iu\,\tau$
inside its $C$ function (and nothing else in it depends on $r$). Our
`merton_char_func`, standalone, carries drift $(r - q - \lambda\kappa_J)$,
including the jump compensator $-\lambda\kappa_J$.

If we naively multiplied the two *full* characteristic functions, the $(r-q)$
drift would appear twice, once from each, and the product would carry $2(r-q)$
of drift. Not a martingale.

The martingale condition tells us exactly how much drift the assembled function
may carry. Every model must satisfy

$$\phi(-i) = \mathbb{E}[S_T] = S_0\,e^{(r-q)\tau},$$

because at $u = -i$ the transform $e^{iu\ln S_T}$ becomes $e^{\ln S_T} = S_T$.
So the total drift must be exactly $(r-q)$ from the diffusion side, and the jump
side must be *compensated* by $-\lambda\kappa_J$ so the jumps add no net growth.

**The clean assembly.** Take Heston's characteristic function carrying the
$(r-q)$ diffusion drift, and multiply by the pure jump factor (no drift) *and* an
explicit compensator term:

$$\phi_{\text{Bates}}(u) = \phi_{\text{Heston}}^{(r-q)}(u)
\cdot \underbrace{e^{iu(-\lambda\kappa_J\tau)}}_{\text{compensator, once}}
\cdot \underbrace{\exp\!\Big(\lambda\tau\big[e^{iu\mu_J - \frac12 u^2\delta_J^2} - 1\big]\Big)}_{\text{jump factor}}.$$

Three factors, each carrying exactly one thing:

- **Heston** carries the $(r-q)$ diffusion drift and the stochastic-vol dynamics.
- **The compensator** $e^{iu(-\lambda\kappa_J\tau)}$ carries the jump-drift
  correction, added once because Heston has no jump knowledge.
- **The jump factor** carries the pure jump randomness, no drift.

So $(r-q)$ appears once and $-\lambda\kappa_J$ appears once. Martingale-consistent
by construction, and the check $\phi(-i) = S_0 e^{(r-q)\tau}$ confirms it.

**Reusing the code.** `heston_char_func` is reused directly, with $(r-q)$ passed
into its $r$-slot (legal because its drift is a single $r\,iu\,\tau$ term with
nothing else riding on $r$). `merton_char_func` is *not* reused wholesale, since
its built-in $(r-q)$ drift would double-count against Heston; only Merton's jump
factor plus the explicit compensator are used.

## 4. The cumulants

The COS truncation range $[a, b]$ needs the first two cumulants of $\log S_T$.
Cumulants live in log-characteristic-function space: since the characteristic
functions multiply, their logs add, and the cumulants (Taylor coefficients of
$\log\phi$) add term by term. So the Bates cumulants are the Heston cumulants plus
the jump cumulants:

$$c_1 = c_1^{\text{Heston}}(r-q) + \lambda\tau\mu_J - \lambda\kappa_J\tau$$

$$c_2 = c_2^{\text{Heston}} + \lambda\tau(\mu_J^2 + \delta_J^2)$$

The $-\lambda\kappa_J\tau$ in $c_1$ is the compensator shifting the mean; the
$\lambda\tau(\mu_J^2 + \delta_J^2)$ in $c_2$ is the jump variance via the
compound-Poisson second moment $\mathbb{E}[Y^2] = \mu_J^2 + \delta_J^2$ (the
$\mu_J^2$ term is easy to drop and doing so narrows the range and loses wing
accuracy). Both reuse the Heston cumulants with $(r-q)$, plus the same Merton jump
terms as Week 7.

Because Bates carries *both* stochastic-vol fat tails and jump fat tails, it is
more truncation-hungry than either alone. Expect to need a wider range $L$ and
more terms $N$ than Heston or Merton, the same lesson Kou taught when its
double-exponential tails demanded a wider $L$. The needed values are validated,
not assumed.

## 5. Why Bates should succeed where the halves failed

The empirical motivation is concrete, from the earlier weeks:

- **Heston alone** fits the term structure of volatility but, being Markovian and
  mean-reverting, flattens the smile too fast at short maturities. It cannot make
  a 29 DTE skew as steep as the market prices.
- **Jumps alone** (Merton, Kou on a constant-vol diffusion) make steep short-dated
  skew, but on real SPX could only reach it by degenerating (Kou's up-jump
  probability collapsed to near zero, a one-sided fit), and they carry no term
  structure of their own.

Bates has both ingredients. The stochastic variance handles how the smile evolves
across maturities; the jumps inject the short-dated tail weight that the diffusion
lacks. The expectation is that Bates fits the whole surface (short and long
maturities together) better than any of its constituents alone, and with less
degenerate parameters than the pure jump models needed.

Whether it delivers, and whether the eight parameters are identifiable from the
data or just partly redundant, is the calibration question. As always, more
parameters fit better almost by definition, so the honest test is whether the
improvement is meaningful and the fitted parameters are sensible, not whether the
number goes down.

In [1]:
# Imports. Reuse the certified pieces; Bates is a composition.
import numpy as np
import matplotlib.pyplot as plt

# from models.bates import bates_char_func, bates_cumulants   # the new model file
# from pricing.fourier import bates_cos_price                 # wrapper over the COS core
# from models.heston import heston_char_func                  # reused as the diffusion part
# from pricing.fourier import cos_call_price                  # Heston COS, for the lam->0 check

In [ ]:
# Validation, ground-truth-first, mirroring Merton/Kou plus a Heston-limit check:
#
# Gate 1: lam -> 0 recovers Heston.
#   bates_cos_price with lam=0 must equal cos_call_price (Heston COS) to
#   truncation accuracy. The jump factor and compensator both vanish at lam=0.
#
# Gate 2: martingale.
#   bates_char_func(-i) = S0 exp((r-q)tau). Certifies (r-q) appears once and
#   -lam*kappa_j once (the compensator placement).
#
# Gate 3: phi(0) = 1.
#
# Gate 4: xi -> 0, v0 = theta reduces to Merton.
#   With zero vol-of-vol and variance pinned at its mean, the Heston part becomes
#   a constant-vol diffusion at vol sqrt(theta), so Bates should match a Merton
#   priced at sigma = sqrt(theta). A second, independent limit that exercises the
#   jump side while collapsing the stochastic-vol side.
#
# Gate 5: COS vs Monte Carlo.
#   Simulate Bates paths (Heston QE variance + compound Poisson jumps), price by
#   MC with its standard error, confirm COS within a few SE (z-score). The
#   independent ground truth for the composed char func.
#
# Truncation: sweep N and L. Bates needs more than Heston/Merton alone (both
# fat-tail sources present). Confirm parity residual and MC agreement tighten
# with wider L, as Kou showed.

## 6. Tomorrow's plan

1. Write `bates_char_func` and `bates_cumulants` in `models/bates.py` (the
   composition above), and the `bates_cos_price` wrapper in `pricing/fourier.py`.
2. Validation gates in order (Cell 8): Heston limit ($\lambda\to0$), martingale,
   $\phi(0)=1$, Merton limit ($\xi\to0$, $v_0=\theta$), then COS vs Monte Carlo.
   Two independent reduction limits (Heston and Merton) is a stronger test than
   either alone, since Bates must collapse correctly to both of its parents.
3. Truncation sweep: find the $N, L$ Bates needs given both fat-tail sources.
4. Later in the week: Variance Gamma (a pure-jump, infinite-activity contrast),
   then cross-model benchmarking (Heston vs Merton vs Kou vs Bates vs VG on the
   same surface).